In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-terrain2-kp2000kd50-relinvel20'  # ckpt = 20000
# exp_name = 'friction-walking-terrain1-kp4000kd50-linvel20-correct0.1-angvel4-plus-0.5'  # ckpt = 20000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand'
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.0
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.0,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.1771, -0.0822, -0.3819,  0.1484, -1.1932,  0.1420,  0.0718,  0.1889,
          0.5174,  0.1196, -1.0868,  0.4054]], device='cuda:0')
Scaled actions :  tensor([[-0.1771, -0.0822, -0.3819,  0.1484, -1.1932,  0.1420,  0.0718,  0.1889,
          0.5174,  0.1196, -1.0868,  0.4054]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-7.5848e-05, -1.8629e-02, -2.9482e-04,  2.0463e-05, -5.1222e-08,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -9.1317e-05,
          4.6075e-06, -1.3769e-05,  7.5769e-04, -1.5931e-02, -7.4130e-04,
          7.2015e-05,  6.0907e-07, -1.6510e-05,  7.4625e-04, -1.5504e-02,
          3.6472e-04, -1.0281e-03,  3.2452e-04,  1.9499e-02,  2.1739e-02,
         -1.2401e+00, -2.5371e-02,  6.1505e-04, -4.9496e-05,  1.9323e-02,
          2.1482e-02, -1.2254e+00,  1.2508e-02, -1.7705e-01, -8.2219e-02,
         -3.8193e-01,  1.4836e-01, -1.1932e+00,  1.4199e-01,  7.1801e-02,
          1.8885e-01,  5.1738e-01,  1.1963e-01, -1.0868e+00,  4.0544e-01]],
       device='cuda:0')
torques: [  2.09592097   0.17792792   8.30804663 -11.84101228 -38.19172146
  -0.68532763  -1.85606517  -0.07286018   8.22259317 -11.71486312
 -37.7223969    0.37270003]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.1408, -0.5356, -0.0771,  0.2773, -0.5971,  0.0753,  0.1624,  0.7592,
         -0.0275,  0.3451, -0.8288,  1.0364]], device='cuda:0')
Scaled actions :  tensor([[-0.1408, -0.5356, -0.0771,  0.2773, -0.5971,  0.0753,  0.1624,  0.7592,
         -0.0275,  0.3451, -0.8288,  1.0364]], device='cuda:0')
obs :  tensor([[-1.2608e-01, -2.2632e-01,  4.8444e-01, -6.8032e-03,  1.0098e-03,
         -9.9998e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -7.3106e-02,
         -5.6662e-03, -2.3778e-02,  1.0781e-01, -1.7860e-01, -3.0963e-01,
          3.3342e-02, -1.6799e-02,  9.5422e-03,  8.4020e-02, -2.8031e-01,
          5.4098e-01, -1.5441e-01, -1.5871e-01, -1.3797e-01,  6.7593e-01,
         -2.6861e+00,  3.7001e+00,  6.2283e-03,  1.5636e-02,  1.9337e-01,
          3.6495e-01, -3.9566e+00,  1.3543e+00, -1.4076e-01, -5.3563e-01,
         -7.7076e-02,  2.7732e-01, -5.9709e-01,  7.5295e-02,  1.6236e-01,
          7.5922e-01, -2.7504e-02,  3.4511e-01, -8.2882e-01,  1.0

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.9277, -1.1661, -0.5927, -1.1072,  0.7540, -1.4234,  0.7192,  0.1666,
         -1.0917, -1.0932,  0.2830,  1.0510]], device='cuda:0')
Scaled actions :  tensor([[ 0.9277, -1.1661, -0.5927, -1.1072,  0.7540, -1.4234,  0.7192,  0.1666,
         -1.0917, -1.0932,  0.2830,  1.0510]], device='cuda:0')
obs :  tensor([[-1.9953e-02, -8.9120e-03,  1.2362e-01, -1.0034e-02,  3.7658e-03,
         -9.9994e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -9.4402e-02,
         -4.3040e-02, -4.3150e-02,  1.9762e-01, -3.5589e-01,  3.3835e-02,
          7.0658e-02, -2.6747e-03,  2.0131e-02,  1.7111e-01, -6.6430e-01,
          5.1027e-01, -6.1175e-02, -2.4911e-01, -1.7485e-02,  3.0854e-01,
         -2.4188e+00,  1.8506e+00,  2.8075e-01,  6.2311e-02,  5.3173e-02,
          3.7887e-01, -3.3785e+00,  1.6896e+00,  9.2774e-01, -1.1661e+00,
         -5.9273e-01, -1.1072e+00,  7.5399e-01, -1.4234e+00,  7.1925e-01,
          1.6656e-01, -1.0917e+00, -1.0932e+00,  2.8298e-01,  1.0

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 0.0869,  0.6854,  0.8734, -0.6264,  0.8360, -0.0637,  0.1807,  0.5422,
          0.4077, -0.8009,  0.2057, -0.0369]], device='cuda:0')
Scaled actions :  tensor([[ 0.0869,  0.6854,  0.8734, -0.6264,  0.8360, -0.0637,  0.1807,  0.5422,
          0.4077, -0.8009,  0.2057, -0.0369]], device='cuda:0')
obs :  tensor([[-0.0882,  0.8209, -1.1551,  0.0080,  0.0068, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0147, -0.0794, -0.0783,  0.2326,  0.0559, -0.5793,  0.2313,
          0.0246, -0.0291,  0.2398, -0.2890,  0.5392,  0.8373, -0.2285, -0.2070,
          0.0393,  2.4521, -4.0399,  1.1467,  0.0831, -0.4326,  0.2202,  3.9637,
          2.5897,  0.0869,  0.6854,  0.8734, -0.6264,  0.8360, -0.0637,  0.1807,
          0.5422,  0.4077, -0.8009,  0.2057, -0.0369]], device='cuda:0')
torques: [ 200.         -200.         -200.         -200.          200.
 -200.           16.54165912  100.40429582 -200.         -200.
  200.          200.        ]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.9999, -0.9978,  0.6626,  1.1421, -0.8430,  1.4598, -0.6805, -0.6967,
          1.1442, -0.7859, -0.5344, -0.7943]], device='cuda:0')
Scaled actions :  tensor([[-0.9999, -0.9978,  0.6626,  1.1421, -0.8430,  1.4598, -0.6805, -0.6967,
          1.1442, -0.7859, -0.5344, -0.7943]], device='cuda:0')
obs :  tensor([[-0.6190, -0.2620,  0.2327,  0.0163,  0.0225, -0.9996,  1.0000,  0.0000,
          0.0000,  0.0995, -0.1194, -0.0626,  0.2191,  0.2989, -0.2349,  0.3128,
          0.0943, -0.0370,  0.2379,  0.0852,  0.4476,  0.2642, -0.0605,  0.2977,
         -0.2761,  2.2041,  1.8573, -0.0475,  0.4928,  0.2736, -0.2673,  1.6958,
          0.8520, -0.9999, -0.9978,  0.6626,  1.1421, -0.8430,  1.4598, -0.6805,
         -0.6967,  1.1442, -0.7859, -0.5344, -0.7943]], device='cuda:0')
torques: [-176.2506455   200.          200.         -200.          200.
  200.         -163.10533757  200.          200.         -200.
   78.50763069  200.        ]
データ収集: step 6


In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[-1.1069, -0.2458, -0.8312,  0.3713,  0.2851, -0.9330,  0.0141, -0.5410,
          0.1564,  0.3638,  0.4808, -0.6295]], device='cuda:0')
Scaled actions :  tensor([[-1.1069, -0.2458, -0.8312,  0.3713,  0.2851, -0.9330,  0.0141, -0.5410,
          0.1564,  0.3638,  0.4808, -0.6295]], device='cuda:0')
obs :  tensor([[ 0.1498, -1.0471,  1.7334, -0.0103,  0.0296, -0.9995,  1.0000,  0.0000,
          0.0000,  0.0542, -0.1710,  0.0356,  0.1730, -0.2181,  0.5136,  0.1340,
          0.1654,  0.1092,  0.1550,  0.0497,  0.1234, -0.8048, -0.3351,  0.5622,
         -0.2721, -2.8295,  1.2188, -1.4637,  0.0958,  1.0823, -0.6396,  1.3883,
          0.2697, -1.1069, -0.2458, -0.8312,  0.3713,  0.2851, -0.9330,  0.0141,
         -0.5410,  0.1564,  0.3638,  0.4808, -0.6295]], device='cuda:0')
torques: [-200.         -200.          200.          200.          -81.58136492
 -200.         -167.05248205 -200.          200.         -200.
  200.          200.        ]
データ収集:

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.0586,  0.4432, -1.7659,  0.1118,  0.2610, -0.5894,  1.3451,  0.2702,
         -0.8423,  0.3950,  0.5496, -0.2106]], device='cuda:0')
Scaled actions :  tensor([[ 0.0586,  0.4432, -1.7659,  0.1118,  0.2610, -0.5894,  1.3451,  0.2702,
         -0.8423,  0.3950,  0.5496, -0.2106]], device='cuda:0')
obs :  tensor([[ 0.3650,  0.0805,  1.4602, -0.0259,  0.0215, -0.9994,  1.0000,  0.0000,
          0.0000, -0.2278, -0.2356,  0.0311,  0.1693, -0.0768,  0.0346, -0.0566,
          0.1563,  0.2430,  0.0815,  0.0168,  0.0351, -1.7692, -0.3724, -0.4585,
          0.1333,  1.5347, -1.3847, -0.4180, -0.1351,  0.2533, -0.1621,  1.5336,
          1.4633,  0.0586,  0.4432, -1.7659,  0.1118,  0.2610, -0.5894,  1.3451,
          0.2702, -0.8423,  0.3950,  0.5496, -0.2106]], device='cuda:0')
torques: [-200.          -46.09162494 -200.          200.          200.
  200.          200.         -200.         -200.          200.
  200.          117.11502641]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 1.0138,  0.1322,  0.9898,  0.2768, -0.6246,  0.5859, -1.0065, -0.6353,
          1.3089, -0.6259,  1.9457, -0.8361]], device='cuda:0')
Scaled actions :  tensor([[ 1.0138,  0.1322,  0.9898,  0.2768, -0.6246,  0.5859, -1.0065, -0.6353,
          1.3089, -0.6259,  1.9457, -0.8361]], device='cuda:0')
obs :  tensor([[ 8.1812e-02,  1.2080e+00, -1.7136e-01,  1.6756e-03,  1.5937e-02,
         -9.9987e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -4.6572e-01,
         -2.6946e-01, -8.3214e-02,  1.9369e-01,  2.1311e-02, -5.7288e-02,
          2.0552e-02,  1.7698e-01,  2.0385e-01,  1.0154e-01,  2.7805e-02,
          9.1765e-02, -8.6348e-01,  4.7148e-02, -7.2137e-01,  6.1075e-02,
          2.5812e+00, -2.5591e+00,  1.1979e+00,  1.6660e-01, -6.5934e-01,
          2.8394e-01,  1.8121e+00,  3.3832e+00,  1.0138e+00,  1.3221e-01,
          9.8982e-01,  2.7680e-01, -6.2462e-01,  5.8587e-01, -1.0065e+00,
         -6.3530e-01,  1.3089e+00, -6.2593e-01,  1.9457e+00, -8.3

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[ 0.1796,  0.4205,  0.9917,  0.4913, -0.1161, -0.1753,  0.6794,  0.4037,
          1.0131, -0.4333,  0.3192, -0.1415]], device='cuda:0')
Scaled actions :  tensor([[ 0.1796,  0.4205,  0.9917,  0.4913, -0.1161, -0.1753,  0.6794,  0.4037,
          1.0131, -0.4333,  0.3192, -0.1415]], device='cuda:0')
obs :  tensor([[-0.1158,  0.1141,  0.5496, -0.0237,  0.0409, -0.9989,  1.0000,  0.0000,
          0.0000, -0.0355, -0.0309, -0.1899,  0.1765, -0.6395, -0.0970,  0.0358,
          0.0686,  0.1241,  0.1071, -0.6302,  0.3332, -0.0398, -0.0499, -0.2111,
          0.0739, -0.5159, -0.0393, -0.0108,  0.0440,  0.0592,  0.0122, -0.4768,
          0.0056,  0.1796,  0.4205,  0.9917,  0.4913, -0.1161, -0.1753,  0.6794,
          0.4037,  1.0131, -0.4333,  0.3192, -0.1415]], device='cuda:0')
torques: [-200.         -200.         -200.         -200.          200.
 -113.09146167  -74.41106969 -200.         -200.         -200.
  200.         -200.        ]
データ収集: step 10

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.5847, -1.1520, -0.6334,  0.1023, -0.1365,  0.0530, -0.1708, -1.2560,
         -1.4866, -1.2001, -0.8583,  0.1343]], device='cuda:0')
Scaled actions :  tensor([[-0.5847, -1.1520, -0.6334,  0.1023, -0.1365,  0.0530, -0.1708, -1.2560,
         -1.4866, -1.2001, -0.8583,  0.1343]], device='cuda:0')
obs :  tensor([[-0.3595, -0.1715,  0.4223, -0.0239,  0.0499, -0.9985,  1.0000,  0.0000,
          0.0000, -0.0326, -0.0321, -0.2229,  0.1990, -0.7305, -0.1114,  0.0445,
          0.0894,  0.1445,  0.1051, -0.7149,  0.3220,  0.0582,  0.0298, -0.1266,
          0.1437, -0.4054, -0.0899,  0.0882,  0.1534,  0.1367, -0.0285, -0.3799,
         -0.1062, -0.5847, -1.1520, -0.6334,  0.1023, -0.1365,  0.0530, -0.1708,
         -1.2560, -1.4866, -1.2001, -0.8583,  0.1343]], device='cuda:0')
torques: [ 200.          200.          200.          200.          200.
 -134.71815853  200.          200.          200.         -200.
  200.         -200.        ]
データ収集: step 1

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.1319,  0.4363,  1.2082,  0.4302,  0.2285, -0.1108,  0.6589,  0.0573,
          0.7795, -0.7100,  0.6636, -0.3582]], device='cuda:0')
Scaled actions :  tensor([[ 0.1319,  0.4363,  1.2082,  0.4302,  0.2285, -0.1108,  0.6589,  0.0573,
          0.7795, -0.7100,  0.6636, -0.3582]], device='cuda:0')
obs :  tensor([[-0.1722,  0.0867,  0.5684, -0.0245,  0.0620, -0.9978,  1.0000,  0.0000,
          0.0000, -0.0318, -0.0349, -0.2564,  0.2186, -0.7993, -0.1171,  0.0500,
          0.1127,  0.1641,  0.0996, -0.7915,  0.2902, -0.0400, -0.0496, -0.2005,
          0.0606, -0.2941,  0.0215, -0.0219,  0.0859,  0.0649, -0.0250, -0.3667,
         -0.1923,  0.1319,  0.4363,  1.2082,  0.4302,  0.2285, -0.1108,  0.6589,
          0.0573,  0.7795, -0.7100,  0.6636, -0.3582]], device='cuda:0')
torques: [-200.         -200.         -200.         -200.          200.
  200.         -200.         -200.         -200.         -200.
  200.         -134.52972772]
データ収集: step 1

In [55]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=1.287, Scaled action max=1.287
Step 1/10, Total steps: 322
steps: 322
actions : tensor([[-0.7229, -0.7316, -0.4623,  0.4050, -0.2916, -0.2572, -0.0343, -0.4235,
         -0.0391, -0.6942,  1.2871, -0.6971]], device='cuda:0')
target_dof_pos: tensor([[-7.9720e-01, -7.3247e-01, -1.1960e+00,  1.9738e+00, -1.0346e+00,
         -2.3244e-01, -6.7451e-02, -3.6122e-01, -1.0365e+00,  8.6009e-01,
          2.4146e-04, -5.1981e-01]], device='cuda:0')
Step 1: Original action max=1.900, Scaled action max=1.900
Step 2: Original action max=2.476, Scaled action max=2.476
データ収集完了: 10 steps collected with action_scale=1.0


In [49]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [50]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
